In [1]:
import pandas as pd
from pathlib import Path


# =========================================================
# Paths
# =========================================================

# IMPORTANT:
# This must be the SAME split_dir used when training the flow.
SPLIT_DIR = Path(
    "/vol/biomedic3/tx1215/mamo-flow/assets/embed_splits_v1"
)

OUT_PATH = (
    SPLIT_DIR
    / "EMBED_same_breast_CC_MLO_pairs.csv"
)


# =========================================================
# 1. Load train / valid / test CSVs
# =========================================================

dfs = []

for split in ["train", "valid", "test"]:

    split_path = SPLIT_DIR / f"{split}.csv"

    df_split = pd.read_csv(
        split_path,
        low_memory=False,
    )

    df_split["split"] = split

    dfs.append(df_split)


df = pd.concat(
    dfs,
    ignore_index=True,
)

print("Total images:", len(df))

print(
    df["split"].value_counts()
)


# =========================================================
# 2. Normalize view / laterality
# =========================================================

df["ViewPosition_norm"] = (
    df["ViewPosition"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df["Laterality_norm"] = (
    df["ImageLateralityFinal"]
    .astype(str)
    .str.upper()
    .str.strip()
)


# =========================================================
# 3. Keep only standard 2D CC / MLO
# =========================================================

# cview was already encoded during split generation:
# 0 = 2D
# 1 = CView
if "cview" in df.columns:
    df = df[
        df["cview"] == 0
    ].copy()


# Remove spot magnification if present
if "spot_mag" in df.columns:

    df = df[
        df["spot_mag"]
        .fillna(0)
        .astype(str)
        .isin(
            [
                "0",
                "0.0",
                "False",
                "false",
            ]
        )
    ].copy()


# Standard left/right CC + MLO only
df_std = df[
    df["Laterality_norm"].isin(
        ["L", "R"]
    )
    &
    df["ViewPosition_norm"].isin(
        ["CC", "MLO"]
    )
].copy()


print(
    "Standard CC/MLO images:",
    len(df_std),
)


# =========================================================
# 4. Build same-patient / same-exam / same-breast pairs
# =========================================================

# Include split explicitly.
# Patients were split before this stage, so CC/MLO pair
# should naturally belong to the same split.

group_cols = [
    "split",
    "empi_anon",
    "acc_anon",
    "Laterality_norm",
]


pairs = []

skipped_missing = 0
skipped_ambiguous = 0


for (split, empi, acc, lat), g in df_std.groupby(
    group_cols
):

    cc_rows = g[
        g["ViewPosition_norm"] == "CC"
    ]

    mlo_rows = g[
        g["ViewPosition_norm"] == "MLO"
    ]


    # -----------------------------------------------------
    # Need both CC and MLO
    # -----------------------------------------------------

    if (
        len(cc_rows) == 0
        or len(mlo_rows) == 0
    ):
        skipped_missing += 1
        continue


    # -----------------------------------------------------
    # For GT comparison only keep unambiguous pairs
    #
    # exactly 1 CC + exactly 1 MLO
    # -----------------------------------------------------

    if (
        len(cc_rows) != 1
        or len(mlo_rows) != 1
    ):
        skipped_ambiguous += 1
        continue


    cc = cc_rows.iloc[0]
    mlo = mlo_rows.iloc[0]


    # -----------------------------------------------------
    # Basic pair information
    # -----------------------------------------------------

    pair = {

        "split": split,

        "empi_anon": empi,
        "acc_anon": acc,
        "laterality": lat,

        # Exact image paths used by EMBED loader
        "cc_path": cc["image_path"],
        "mlo_path": mlo["image_path"],

        # Very useful:
        # exact dataset indices inside each split
        "cc_cache_idx": int(
            cc["cache_idx"]
        ),

        "mlo_cache_idx": int(
            mlo["cache_idx"]
        ),
    }


    # -----------------------------------------------------
    # Save ALL metadata from the split CSV
    #
    # This includes the already processed:
    #
    # age
    # view
    # density
    # scanner
    # cview
    #
    # as well as original EMBED metadata.
    # -----------------------------------------------------

    for col in df.columns:

        pair[f"cc_{col}"] = cc[col]
        pair[f"mlo_{col}"] = mlo[col]


    pairs.append(pair)


# =========================================================
# 5. Save paired dataframe
# =========================================================

pairs_df = pd.DataFrame(
    pairs
)


pairs_df.to_csv(
    OUT_PATH,
    index=False,
)


# =========================================================
# 6. Summary
# =========================================================

print()
print(
    "Number of valid CC/MLO pairs:",
    len(pairs_df),
)

print(
    "Skipped - missing CC or MLO:",
    skipped_missing,
)

print(
    "Skipped - ambiguous multiple views:",
    skipped_ambiguous,
)

print()
print("Pairs per split:")

print(
    pairs_df["split"]
    .value_counts()
)

print()
print(
    "Saved to:",
    OUT_PATH,
)


pairs_df.head()

Total images: 297373
split
train    237746
test      36934
valid     22693
Name: count, dtype: int64
Standard CC/MLO images: 297373

Number of valid CC/MLO pairs: 81581
Skipped - missing CC or MLO: 1858
Skipped - ambiguous multiple views: 39656

Pairs per split:
split
train    65370
test      9998
valid     6213
Name: count, dtype: int64

Saved to: /vol/biomedic3/tx1215/mamo-flow/assets/embed_splits_v1/EMBED_same_breast_CC_MLO_pairs.csv


,split,empi_anon,acc_anon,laterality,cc_path,mlo_path,cc_cache_idx,mlo_cache_idx,cc_empi_anon,mlo_empi_anon,...,cc_cview,mlo_cview,cc_age,mlo_age,cc_split,mlo_split,cc_ViewPosition_norm,mlo_ViewPosition_norm,cc_Laterality_norm,mlo_Laterality_norm
0,test,10000879,6992096043050201,L,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,14592,14590,10000879,10000879,...,0,0,41.879026,41.879026,test,test,CC,MLO,L,L
1,test,10000879,6992096043050201,R,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,14591,14589,10000879,10000879,...,0,0,41.879026,41.879026,test,test,CC,MLO,R,R
2,test,10015693,1334581155737139,L,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,8940,8939,10015693,10015693,...,0,0,65.167663,65.167663,test,test,CC,MLO,L,L
3,test,10015693,1334581155737139,R,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,8938,8937,10015693,10015693,...,0,0,65.167663,65.167663,test,test,CC,MLO,R,R
4,test,10015693,2281263876413228,L,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,/vol/biodata/data/Mammo/EMBED/pngs/1024x768/co...,14550,14549,10015693,10015693,...,0,0,67.407271,67.407271,test,test,CC,MLO,L,L
